# Inspect replay `.pt` dumps

`.pt` files are PyTorch binary checkpoints/tensors — open them here to see
episode lengths, rewards, continue flags, and actual Crafter frames.

Default path: `data/m3_replay.pt` (from `scripts/collect_replay.py`).


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from training.replay_buffer import ReplayBuffer

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

%matplotlib inline

REPLAY_PATH = Path("data/m3_replay.pt")  # or data/m3_replay_smoke.pt
assert REPLAY_PATH.exists(), f"Missing {REPLAY_PATH} — run: python scripts/collect_replay.py"

raw = torch.load(REPLAY_PATH, weights_only=False)
print(f"file: {REPLAY_PATH}  ({REPLAY_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"keys: {list(raw.keys())}")
print(f"episodes: {len(raw['episodes'])}  total_steps: {raw['total_steps']}")


## What’s inside one episode

Each episode is a dict of aligned length-`T` tensors:

| key | shape | meaning |
|---|---|---|
| `obs` | `[T, 64, 64, 3]` uint8 | Crafter frames |
| `actions` | `[T]` int64 | action taken after that frame |
| `rewards` | `[T]` float | env reward at that step |
| `cont` | `[T]` float | `1` = episode continues, `0` = terminated |


In [ ]:
ep = raw["episodes"][0]
for k, v in ep.items():
    t = v if hasattr(v, "shape") else torch.as_tensor(v)
    print(f"{k:8s} shape={tuple(t.shape)} dtype={t.dtype}")

lengths = np.array([e["obs"].shape[0] for e in raw["episodes"]])
rewards = torch.cat([e["rewards"].float() for e in raw["episodes"]])
conts = torch.cat([e["cont"].float() for e in raw["episodes"]])

print(f"\nlengths: min={lengths.min()} median={np.median(lengths):.0f} max={lengths.max()}")
print(
    f"reward: mean={rewards.mean():.4f}  nonzero={(rewards != 0).float().mean():.2%}  "
    f"sum={rewards.sum():.1f}"
)
print(f"continue mean: {conts.mean():.3f}  (near 1.0 is normal — deaths are rare)")


## Episode length + reward histograms


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].hist(lengths, bins=15, color="#264653")
axes[0].set_title("episode lengths")
axes[0].set_xlabel("steps")

axes[1].hist(rewards.numpy(), bins=30, color="#e76f51")
axes[1].set_title("per-step rewards (all episodes)")
axes[1].set_xlabel("reward")

ep_rew = [float(e["rewards"].sum()) for e in raw["episodes"]]
axes[2].bar(range(len(ep_rew)), ep_rew, color="#2a9d8f")
axes[2].set_title("total reward per episode")
axes[2].set_xlabel("episode index")

fig.tight_layout()
plt.show()


## Browse frames from one episode

Change `EP` / `START` to scrub around. Sparse non-zero rewards are marked.


In [ ]:
EP = 0
START = 0
N_SHOW = 8
STRIDE = 5  # show every STRIDE-th frame

ep = raw["episodes"][EP]
obs = ep["obs"]
rew = ep["rewards"]
act = ep["actions"]
cont = ep["cont"]
T = obs.shape[0]

idxs = list(range(START, min(T, START + N_SHOW * STRIDE), STRIDE))[:N_SHOW]
fig, axes = plt.subplots(1, len(idxs), figsize=(2.2 * len(idxs), 2.6))
if len(idxs) == 1:
    axes = [axes]
for ax, t in zip(axes, idxs):
    ax.imshow(obs[t].numpy())
    ax.set_title(f"t={t}\na={int(act[t])} r={float(rew[t]):.2f}\nc={float(cont[t]):.0f}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"episode {EP}  (len={T})", y=1.05)
fig.tight_layout()
plt.show()

nonzero = (rew != 0).nonzero(as_tuple=False).flatten().tolist()
print(f"non-zero reward timesteps in ep {EP}: {nonzero[:40]}{'...' if len(nonzero) > 40 else ''}")


## Sample a training window (what the world model actually sees)

Training does **not** feed whole episodes. The sequential replay buffer cuts
contiguous length-`L` chunks so the RSSM can learn dynamics over time.


In [ ]:
buf = ReplayBuffer(seed=0)
buf.load_state_dict(raw)
batch = buf.sample(batch_size=4, seq_len=32)

print("sampled batch shapes:")
for k, v in batch.items():
    print(f"  {k:8s} {tuple(v.shape)} {v.dtype}")

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for b in range(4):
    for j, t in enumerate(range(0, 32, 4)):
        axes[b, j].imshow(batch["obs"][b, t].numpy())
        axes[b, j].set_title(f"b{b} t={t}", fontsize=7)
        axes[b, j].axis("off")
fig.suptitle("4 contiguous seq_len=32 windows (every 4th frame)")
fig.tight_layout()
plt.show()
